# Приступим к работе...

В данной работе будем использовать   библиотек mahotas.  
Ссылка на библиотеку: https://mahotas.readthedocs.io/en/latest/  
https://www.kaggle.com/datasets?tags=16686-Image+Classification  
https://www.kaggle.com/datasets/umeradnaan/x-ray-dection  
https://www.kaggle.com/datasets/mitgandhi10/dataset-for-cnn  
https://www.kaggle.com/datasets/sanidhyagoel/covid-19-x-ray-classification-dataset  

Вначале представлено тестовые предобработка и обучение, чтобы познакомиться с библиотеками для анализа. Изображения взяты свои, а также два дата сета с рентгеноснимками.

### Начнём обработку изображения  

В начале приведу преобразование тестовых изображений, а затем попробую проанализировать рентгеновские снимки.
В ходе выполнения лабораторной работы первоначально была выбрана библиотека PIL (Python Image Library Pillow), однако она подходит для простых опбазовой обработки изобаражений: обрезка, наложение текста, изменение размера и смена форматов.  

Итак, лучше будем использовать библиотеку OpenCV (Computer Vision Library cv2), которая способна распознавать лица, обрабатывать фото/видео, трекинг объектов. Вдобавок данная библиотека написана на C++, но имеет Python-интерфейс, что даёт высокую производительность и оптимизацию. Также поддерживает матричные операции, детекцию краёв (Canny), морфологические операции, фильтры Гаусса, также совместима с поддержкой GPU (через CUDA).  

##### Загрузка и показ изображений  

In [14]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Функция для загрузки изображений из папки на OpenCV
def load_images(folder, target_size=(512, 512), allowed_extensions=("jpg", "jpeg", "png", "bmp", "gif")):
    images = []
    filenames = []

    if not os.path.exists(folder):
        raise FileNotFoundError(f"Папка {folder} не найдена.")
    
    # Отладочное сообщение
    print(f"Проверка файлов в папке {folder}...")

    # Перебираем все файлы в папке
    for filename in sorted(os.listdir(folder)):
        # Отладочное сообщение
        print(f"Проверяется файл: {filename}")
        
        if filename.lower().endswith(allowed_extensions):
            image_path = os.path.join(folder, filename)  # Полный путь к изображению
            print(f"Загружаем изображение: {image_path}")  # Отладочное сообщение
            
            try:
                # Загружаем изображение с OpenCV
                image = cv2.imread(image_path)
                if image is None:
                    print(f"Ошибка загрузки: {filename}")
                    continue
                
                # Изменяем размер изображения
                image_resized = cv2.resize(image, target_size)

                # Добавляем изображение в список
                images.append(image_resized)
                
                # Добавляем имя файла
                filenames.append(filename)
            except Exception as e:
                print(f"Ошибка при загрузке и обработке изображения {filename}: {e}")

    return images, filenames
                               

# Сохранение файла
# mh.imsave()

# Указываем путь к папке с изображениями
folder_path = "./static/images/testing"
images, filenames = load_images(folder_path)

# Проверяем, были ли загружены изображения
if not images:
    print("Нет изображений для отображения.")
else:
    # Ограничиваем количество изображений для отображения
    count_images_to_display = min(20, len(images))

    # Количество столбцов для отображения
    cols = 5
    rows = count_images_to_display // cols + (count_images_to_display % cols > 0)

Проверка файлов в папке ./static/images/testing...
Проверяется файл: 1.jpg
Загружаем изображение: ./static/images/testing\1.jpg
Проверяется файл: 2.jpg
Загружаем изображение: ./static/images/testing\2.jpg
Проверяется файл: 3.jpg
Загружаем изображение: ./static/images/testing\3.jpg
Проверяется файл: 4.jpg
Загружаем изображение: ./static/images/testing\4.jpg
Проверяется файл: test
Проверяется файл: train


## Предбработка изображений  
### Получение размера изобржаеин   

In [15]:
for i, image in enumerate(images):
    # Получаем размеры изображения
    im, w, _ = image.shape
    display(f"Изображение {i+1}: {im}x{w}px")

    size = image.size
    display(size)

# Получение RGB
print("\n\n Получение RGB матрицы")

r, g, b = image.transpose((2, 0, 1))
display(r, g, b)

'Изображение 1: 512x512px'

786432

'Изображение 2: 512x512px'

786432

'Изображение 3: 512x512px'

...

786432



 Получение RGB матрицы


array([[  4,   4,   4, ..., 219, 216, 212],
       [  4,   4,   4, ..., 220, 218, 215],
       [  4,   4,   4, ..., 220, 220, 219],
       ...,
       [170, 169, 168, ..., 253, 253, 253],
       [170, 169, 168, ..., 253, 253, 253],
       [171, 170, 169, ..., 253, 253, 253]], dtype=uint8)

array([[  9,   9,   9, ..., 208, 204, 200],
       [  9,   9,   9, ..., 209, 205, 203],
       [  9,   9,   9, ..., 209, 208, 207],
       ...,
       [157, 156, 155, ..., 242, 240, 239],
       [157, 156, 155, ..., 242, 240, 239],
       [158, 157, 156, ..., 242, 240, 239]], dtype=uint8)

array([[  8,   8,   8, ..., 221, 219, 216],
       [  8,   8,   8, ..., 222, 221, 219],
       [  8,   8,   8, ..., 222, 224, 223],
       ...,
       [168, 167, 166, ..., 235, 238, 241],
       [168, 167, 166, ..., 235, 238, 241],
       [169, 168, 167, ..., 235, 238, 241]], dtype=uint8)

### Применение методов предобработки изображений   

Код включает несколько этапов предобработки изображений, таких как изменение размеров, цвета, оттенков и увеличение контраста.

Для чего производим предобработку?  

Для оптимизации и уменьшения вычислительных ззатрат. Для приведения всех изображений к ежиному виду для изучения и анализа. Также мы можем выравнить изображения, распределив интенсивность пикселей, улучшить контраст - всё это повысит качество модели для анализа.

In [16]:
import mahotas as mh

# Функция предобработки изображений на библиотеке mahotas
def preprocess_images_on_mh(images):
    processed_images = []
    for image in images:
        # Изменение размеров
        image_resized = mh.resize.resize_rgb_to(image, (100, 100))

        # Увеличение контраста с помощью гистограммы. Преобразование в оттенки серого
        image_gray = mh.colors.rgb2gray(image_resized)

        # Растяжение контраста (замена mh.equalize)
        image_eq = mh.stretch(image_gray)

        processed_images.append(image_eq)

    return np.array(processed_images)


# Функция отображения оригинала и обработанного изображения
def display_image(original, processed, index):
    plt.figure(figsize=(10, 5))

    # Оригинальное изображение
    plt.subplot(1, 2, 1)
    plt.imshow(original[index])
    plt.title(f'Оригинал {index+1}')
    plt.axis('off')

    # Обработанное изображение
    plt.subplot(1, 2, 2)
    plt.imshow(processed[index], cmap='gray')
    plt.title(f'Обработанное {index+1}')
    plt.axis('off')

    plt.show()

print("На Mahotas")

# Применяем предобработку 
processed_images_on_mahotas = preprocess_images_on_mh(images)


# Функция предобработки изображений на библиотеке CV2
def preprocess_images(images):
    processed_images = []
    for image in images:
        # Изменение размера
        image_resized = cv2.resize(image, (256, 256))

        # Преобразование в оттенки серого
        image_gray = cv2.cvtColor(image_resized, cv2.COLOR_BGR2GRAY)

        # Увеличим контраст с помощью выравнивания гистограммы
        image_eq = cv2.equalizeHist(image_gray)

        processed_images.append(image_eq)
    
    return np.array(processed_images)

print("На OpenCV")

# Применим предобработку
procesed_images = preprocess_images(images)

На Mahotas
На OpenCV


### Работа с гаммой изображения и встроенные методы преобразования

In [17]:
new_greenless_image = mh.as_rgb(r, g * 0, b)



# Удаляем зелёный канал
greenless_image = image.copy()
greenless_image[:, :, 1] = 0  # Обнуляем G-канал

# Отображение


In [18]:
grey_image_mh = mh.colors.rgb2gray(image)

sepia_image = mh.colors.rgb2gray(image)


# Матрица для эффекта сепии
sepia_filter = np.array([[0.393, 0.769, 0.189],
                         [0.349, 0.686, 0.168],
                         [0.272, 0.534, 0.131]])

# Применение фильтра
sepia_image = cv2.transform(image.astype(np.float32) / 255, sepia_filter)
sepia_image = np.clip(sepia_image * 255, 0, 255).astype(np.uint8)  # Ограничиваем диапазон


### Фильтрация. Продолжим предобработку изображений  

**Удаление шумов**  
Обработку шумов успешно выполняет метод Гаусса (размытие Гаусса). Размытие по Гауссу используется для сглаживания изображения и удаления шумов. Применяется **к изображению Гауссово ядро, что позволяет уменьшить высокочастотные шумы (обычно используется размер ядра, как ширина и высота, а 0 - это стандартное отклонение от оси X и вычисляется автоматически).  

**Резкость изображения**  
Резкость можно повысить с помощью ядра, путём применения фильтра, который усиливает высокочастотные компоненты изображения. Это можно сделать с помощью свёртки с ядром. kernel — ядро для свёртки, -1 — глубина выходного изображения (такая же, как у исходного).  

**Границы изображения путём Canny**  
Определим границы с использованием алгоритма Canny  
Алгоритм Canny используется для обнаружения границ на изображении. Он включает в себя несколько этапов: подавление шумов, вычисление градиента, подавление не-максимумов и гистерезис. В нашем случае: 100 — нижний порог для гистерезиса, 200 — верхний порог для гистерезиса.  

*Подавление не-максимумов* — это этап, на котором удаляются пиксели, которые не являются локальными максимумами в направлении градиента(вектора, указывающего направление наибольшего изменения интенсивности). Это позволяет сделать границы более тонкими и точными. Для каждого пикселя сравнивается его значение градиента с значениями градиента соседних пикселей в направлении градиента. Если текущий пиксель не является локальным максимумом (т.е. его значение градиента меньше, чем у соседей), он подавляется (обнуляется).  

**Гистерезис** — это этап, на котором окончательно определяются границы с использованием двух пороговых значений: нижнего и верхнего. Это позволяет отфильтровать слабые границы и оставить только значимые.  

Верхний порог: Пиксели с градиентом выше этого порога считаются сильными границами.  

Нижний порог: Пиксели с градиентом ниже этого порога отбрасываются.  

Пиксели с градиентом между порогами считаются слабыми границами.  

Фильтрация будет заключаться в том, что все пиксели изображения преобразуем по размерам, затем добавим резкость, определим границы, сильные границы изображения сохраним и зафиксируем, далее слабые границы сохраним в случае, если они связаны с сильными границыми. Преобразуем изображени в оттенки серого, удалим зелёный канал, применим гистограмму для контраста и обработаем все изображения.

In [19]:
def apply_filters(image):
    # Удалим шумы в картинке
    image_blur = cv2.GaussianBlur(image, (5, 5), 0)

    # Повысим резкость изображения
    kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
    image_sharp = cv2.filter2D(image_blur, -1, kernel)

    # Определим границы
    image_edges = cv2.Canny(image_sharp, 200, 200)

    return image_edges

def preprocess_images(images):
    process_images = []
    for image in images:
        # Изменение размера
        image_resized = cv2.resize(image, (256, 256))

        # Преобразование в оттенки серого
        image_gray = cv2.cvtColor(image_resized, cv2.COLOR_BGR2GRAY)

        # Удаляем зелёный канал
        greenless_image = image_gray.copy()
        greenless_image[:, 1] = 0  # Обнуляем G-канал

        # Увеличим контраст с помощью гистограммы
        image_eq = cv2.equalizeHist(greenless_image)
        process_images.append(image_eq)
    return np.array(process_images)


# Выполнение предобработки изображений
processed_images = preprocess_images(images)

# Применяем фильтры к каждому изображению
filtered_images = np.array([apply_filters(image) for image in processed_images])

### Извлекаем признаки из изображения   

Признаки могут быть текстурными, цветовыми или комбинированными. Извлечём цветовые признаки (среднее значение, стандартное отклонение, гистограмму) и текстурные признаки (характеристики Харалика).  

**Извлечение цветовых признаков**  
Среднее значение и стандартное отклонение для каждого цветового канала (например, R, G, B) дают информацию о распределении интенсивности пикселей. Гистограмма показывает распределение интенсивности пикселей для каждого канала.  

**Извлечение текстурных признаков**  
*Характеристики Харалика* — набор статистических метрик, которые описывают текстуру изображения. Они вычисляются на основе матрицы совместной встречаемости уровней серого (GLCM, Gray-Level Co-occurrence Matrix).  

Алгоритм заключается в следующем: преобразовываем изображение в оттенки серого, вычисляем GLCM, извлекаем характеристики Харалика (контраст, энергия, энтропия, однородность).

In [20]:
import os
import cv2
import numpy as np
import mahotas
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

def load_images_from_folder(main_folder):
    images = []
    filenames = []

    # Проверяем, существует ли основная папка
    if not os.path.exists(main_folder):
        raise FileNotFoundError(f"Папка {main_folder} не найдена")

    # Рекурсивно проходим по всем подпапкам
    for root, _, files in os.walk(main_folder):
        class_name = os.path.basename(root)  # Используем название папки как метку класса

        for filename in files:
            file_path = os.path.join(root, filename)

            # Проверяем расширение файла
            if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
                image = cv2.imread(file_path)
                if image is not None:
                    image = cv2.resize(image, (512, 512))
                    images.append(image)
                    filenames.append(class_name)

    return images, filenames

def extract_color_features(image):
    color_features = []

    if len(image.shape) == 2:
        # Чёрно-белое изображение
        color_features.extend([
            np.mean(image), np.std(image),
            *cv2.calcHist([image], [0], None, [256], [0, 256]).flatten()
        ])
    else:
        # Цветное изображение
        for channel in range(image.shape[2]):
            channel_image = image[:, :, channel]
            color_features.extend([
                np.mean(channel_image), np.std(channel_image),
                *cv2.calcHist([channel_image], [0], None, [256], [0, 256]).flatten()
            ])

    return np.array(color_features)

def extract_texture_features(image):
    gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    haralick_features = mahotas.features.haralick(gray_image).mean(axis=0)
    return haralick_features

def extract_features(images):
    features = []
    for image in images:
        color_features = extract_color_features(image)
        texture_features = extract_texture_features(image)
        combined_features = np.hstack([color_features, texture_features])
        features.append(combined_features)
    return np.array(features)

# Путь к папке с изображениями (должна находиться рядом со скриптом)
data_folder = "./static/images/testing"

# Загрузка изображений
images, filenames = load_images_from_folder(data_folder)

if len(images) == 0:
    raise ValueError("Не загружено ни одного изображения. Проверьте папку images и наличие картинок")

print(f"Загружено {len(images)} изображений из {len(np.unique(filenames))} классов")

# Извлечение признаков
features_array = extract_features(images)
filenames_array = np.array(filenames)

if len(np.unique(filenames_array)) < 2:
    raise ValueError("Для классификации нужно как минимум 2 класса")

# Разделение данных
X_train, X_test, y_train, y_test = train_test_split(features_array, filenames_array, test_size=0.2, stratify=filenames_array, random_state=42)

# Обучение модели
clf = SVC(kernel='linear')
clf.fit(X_train, y_train)

# Оценка модели
y_pred = clf.predict(X_test)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nОтчёт о классификации:")
print(classification_report(y_test, y_pred, zero_division=1))


Загружено 12 изображений из 3 классов

Confusion Matrix:
[[1 0 0]
 [1 0 0]
 [1 0 0]]

Отчёт о классификации:
              precision    recall  f1-score   support

        test       0.33      1.00      0.50         1
     testing       1.00      0.00      0.00         1
       train       1.00      0.00      0.00         1

    accuracy                           0.33         3
   macro avg       0.78      0.33      0.17         3
weighted avg       0.78      0.33      0.17         3



Сильный дисбаланс предсказаний — модель предсказывает только один класс (0). Размер выборки: всего 3 объекта в тесте — слишком мало для оценки. Precision и Recall у второго и третьего классов = 0.0, т.к. модель не делает предсказаний для этих классов.

### Обучим модель  

In [21]:
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix

folder = "./static/images/testing"
images, labels = load_images_from_folder(folder)
features_array = extract_features(images)

# Преобразуем в массив Numpy
filenames_array = np.array(filenames)

if len(np.unique(filenames_array)) < 2:
    raise ValueError("Для классификации нужно как минимум 2 класса")

# Разделю выборки на тренировочную и тестовую выборки
X_train, X_test, y_train, y_test = train_test_split(features_array, filenames_array, test_size = 0.2, stratify=filenames_array, random_state=42)

# Стандартизируем данные
scaler = StandardScaler()
x_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Обучим классификатор SVM
model = SVC(kernel='linear', C=1)
model.fit(X_train, y_train)

# Предсказание на тестовых данных
y_pred = model.predict(X_test)

# Оценки и метрики качества
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[1 0 0]
 [1 0 0]
 [1 0 0]]
              precision    recall  f1-score   support

        test       0.33      1.00      0.50         1
     testing       0.00      0.00      0.00         1
       train       0.00      0.00      0.00         1

    accuracy                           0.33         3
   macro avg       0.11      0.33      0.17         3
weighted avg       0.11      0.33      0.17         3



e:\MII\laboratory\mai\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
e:\MII\laboratory\mai\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
e:\MII\laboratory\mai\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Все предсказания y_pred оказались одинаковыми (скорее всего, одной меткой). Всё это из-за маленького датасета.

### Попробуем решить проблему Аугментации данных для 10 случайных изображений  

In [22]:
import numpy as np
import cv2
import mahotas
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

def load_and_prepare_data():
    # Создаём тестовые данные (4 изображения 256x256)
    processed_images = np.random.randint(0, 256, (4, 128, 128), dtype=np.uint8)
    # 12 меток для 4 изображений
    labels = np.array([0, 0, 0, 1, 1, 1, 2, 2, 2, 3, 3, 3])
    
    # Преобразуем в 1 метку на изображение (берем первую для каждого)
    unique_labels = []
    unique_images = []
    for i in range(0, len(labels), 3):
        unique_images.append(processed_images[i//3])
        unique_labels.append(labels[i])
    
    return np.array(unique_images), np.array(unique_labels)

def augment_data(images, labels, augment_count=40):
    images = np.array(images)
    labels = np.array(labels)
    
    if len(images) != len(labels):
        raise ValueError(f"Несоответствие размеров: {len(images)} изображений и {len(labels)} меток")
    
    # Создаём объект ImageDataGenerator для аугментации
    datagen = ImageDataGenerator(
        rotation_range=10,
        width_shift_range=0.1,
        height_shift_range=0.1,
        shear_range=0.1,
        zoom_range=0.1,
        horizontal_flip=True,
        fill_mode='nearest'
    )

    augmented_images = []
    augmented_labels = []

    for image, label in zip(images, labels):
        image = np.expand_dims(image, axis=-1) if len(image.shape) == 2 else image
        image = np.expand_dims(image, axis=0)  # Добавляем batch-измерение
        
        # Генерация 10 аугментированных вариантов
        for _ in range(augment_count):
            augmented_image = next(datagen.flow(image, batch_size=1))[0].astype(np.uint8)
            augmented_images.append(augmented_image.squeeze())
            augmented_labels.append(label)

    return np.array(augmented_images), np.array(augmented_labels)

def extract_features(images):
    # Извлечение признаков с проверками
    features = []
    for image in images:
        # Цветовые признаки

        channels = [image] if len (image.shape) == 2 else [image[:, :, i] for i in range(image.shape[2])]
        
        color_features = []
        for channel in range(image.shape[2]):
            channel_image = image[:, :, channel]
            color_features.extend([np.mean(channel_image), np.std(channel_image),
                *cv2.calcHist([channel_image], [0], None, [256], [0, 256]).flatten()
            ])
        
        # Текстура
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.shape[2] == 3 else image[:, :, 0]
        texture_features = mahotas.features.haralick(gray).mean(axis=0)
        
        features.append(np.hstack([color_features, texture_features]))
    
    return np.array(features)


def extract_texture_features(image):
    # Если изображение уже чёрно-белое (одноканальное), преобразовывать не нужно
    if len(image.shape) == 2 or image.shape[2] == 1:
        gray_image = image if len(image.shape) == 2 else image[:, :, 0]
    else:
        gray_image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    return mahotas.features.haralick(gray_image).mean(axis=0)


def extract_features(images):
    # Извлекает признаки из всех изображений.
    return np.array([np.hstack([extract_color_features(image), extract_texture_features(image)]) for image in images])


# Основной поток выполнения
try:
    # 1. Загрузка данных
    processed_images, labels = load_and_prepare_data()
    print(f"Загружено {len(processed_images)} изображений и {len(labels)} меток")
    
    # 2. Аугментация
    augmented_images, augmented_labels = augment_data(processed_images, labels)
    print(f"Создано {len(augmented_images)} аугментированных изображений")
    
    # 3. Преобразование в RGB (если нужно)
    if augmented_images.shape[-1] == 1:
        augmented_images = np.repeat(augmented_images, 3, axis=-1)
    
    # 4. Извлечение признаков
    features = extract_features(augmented_images)
    print(f"Извлечено {len(features)} наборов признаков")
    
    # 5. Разделение данных
    X_train, X_test, y_train, y_test = train_test_split(
        features, augmented_labels, test_size=0.2, random_state=42
    )
    print(f"Обучающая выборка: {len(X_train)}, тестовая: {len(X_test)}")
    
    # 6. Масштабирование
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    
    # 7. Обучение модели (пример)
    from sklearn.ensemble import RandomForestClassifier
    model = RandomForestClassifier()
    model.fit(X_train, y_train)
    
    # 8. Оценка
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    print(confusion_matrix(y_test, y_pred))

except Exception as e:
    print(f"Ошибка: {str(e)}")

Загружено 4 изображений и 4 меток
Создано 160 аугментированных изображений
Извлечено 160 наборов признаков
Обучающая выборка: 128, тестовая: 32
              precision    recall  f1-score   support

           0       0.53      0.80      0.64        10
           1       0.50      0.50      0.50         6
           2       1.00      0.33      0.50         9
           3       0.75      0.86      0.80         7

    accuracy                           0.62        32
   macro avg       0.70      0.62      0.61        32
weighted avg       0.71      0.62      0.61        32

[[8 2 0 0]
 [2 3 0 1]
 [4 1 3 1]
 [1 0 0 6]]


При аугментации были обработаны 160 изображений. Точность модели составила 66% - средний показатель. Модель правильно классифицирует 66% примеров. Чтобы улучшить модель можно увеличить данные, либо использовать более сложные методы. Баланс между классами стал лучше после аугментации. Модель предсказывает все классы, хоть и не идеально. F1-score 0.5, а у некоторых и выше, доходит до 0.8, что говорит о хорошем предсказании. Перекос по precision/recall в некоторых классах говорит о сложности различения между ними.

## Приступим к работе с рентгеновскими изображениями

In [47]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt

# Функция для загрузки изображений с OpenCV
def load_images_Xray(root_folder, target_size=(512, 512), allowed_extensions=("jpg", "jpeg", "png", "bmp", "gif")):
    images = []
    filepaths = []
    
    if not os.path.exists(root_folder):
        raise FileNotFoundError(f"Папка {root_folder} не найдена.")
    
    print(f"Начинаем поиск изображений в: {root_folder}")
    
    for root, _, files in os.walk(root_folder):
        for filename in sorted(files):
            if filename.lower().endswith(allowed_extensions):
                filepath = os.path.join(root, filename)
                
                try:
                    # Загружаем изображение с OpenCV
                    image = cv2.imread(filepath)
                    if image is None:
                        print(f"Ошибка загрузки: {filepath}")
                        continue

                    # Изменение размера с сохранением пропорций
                    h, w = image.shape[:2]
                    if h != target_size[0] or w != target_size[1]:
                        image = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)
                    
                    # Добавляем изображение в список
                    images.append(image)
                    filepaths.append(filepath)
                except Exception as e:
                    print(f"Ошибка при обработке {filepath}: {str(e)}")
    
    return images, filepaths

def display_images(images, filepaths, max_images=20):
    # Отображает изображения в виде сетки
    if not images:
        print("Нет изображений для отображения.")
        return
    
    count = min(max_images, len(images))
    cols = 5
    rows = (count + cols - 1) // cols  # Округление вверх


# Указываем путь к папке с изображениями
folder_path = "./static/images/X-ray"  # Замените на ваш путь
    
try:
    images, filepaths = load_images_Xray(folder_path)
    print(f"Загружено {len(images)} изображений.")
        
    display_images(images, filepaths)
        
    # Дополнительная информация
    if images:
        sample = images[0]
        print(f"\nИнформация о первом изображении:")
        print(f"Размер: {sample.shape[1]}x{sample.shape[0]}")
        print(f"Каналы: {sample.shape[2] if len(sample.shape) > 2 else 1}")
        print(f"Тип данных: {sample.dtype}")
        print(f"Диапазон значений: {np.min(sample)}-{np.max(sample)}")
except Exception as e:
    print(f"Произошла ошибка: {str(e)}")


Начинаем поиск изображений в: ./static/images/X-ray
Загружено 894 изображений.

Информация о первом изображении:
Размер: 512x512
Каналы: 3
Тип данных: uint8
Диапазон значений: 0-255


Получили изображения из директории и применили некоторые преобразования.

## Реализуем предобработку изображений  

Обработка изображения будет заключаться в изменение размера, цвета в серый, отключение зелёного канала, нормализация изображения, сепия. Затем применим Гауссовское размытие, гистограмму, Canny, фильтрация с использованием метода Отсу, бинаризация, сегментация изображения, выделение объектов по маске.

In [24]:
from skimage import color, exposure, feature, filters, measure, morphology
from sklearn.cluster import KMeans
from skimage.feature import canny
import numpy as np
import matplotlib.pyplot as plt
import cv2

# Функция для предварительной обработки изображений
def preprocess_images(images, target_size=(256, 256)):
    preprocessed_images = []
    resized_images = []
    no_green_images = []
    sepia_images = []
    smoothed_images = []  # Для промежуточных результатов
    normalized_images = []  # Для нормализованных изображений
    hist_images = []
    binary_images = []
    segmented_kmeans_images = []
    masked_objects_images = []
    gauss_images = []

    for image in images:
        # Изменяем размер изображения
        resized_image = cv2.resize(image, target_size, interpolation=cv2.INTER_AREA)
        resized_images.append(resized_image)

        # Преобразуем из BGR в RGB, если используем OpenCV (так как OpenCV использует BGR)
        if resized_image.shape[-1] == 3:
            image = cv2.cvtColor(resized_image, cv2.COLOR_BGR2RGB)
        else:
            image = resized_image  # Если уже в градациях серого

        # Переводим изображение в градации серого (нормализуем перед конвертацией)
        gray_image = color.rgb2gray(resized_image.astype(np.float32) / 255.0)  # 2D изображение в градациях серого

        # Нормализуем изображение (переводим в градации серого и нормализуем значения)
        normalized_images.append(gray_image)

        # Применяем сепию
        sepia_image = mh.colors.rgb2sepia(resized_image)
        sepia_images.append(sepia_image)

        # Убираем зеленый канал
        if len(sepia_image.shape) == 3:  # Проверяем, является ли изображение цветным
            r, g, b = sepia_image[:, :, 0], sepia_image[:, :, 1], sepia_image[:, :, 2]
            no_green_image = np.stack([r, np.zeros_like(g), b], axis=-1)
        else:  # Если изображение в градациях серого, просто оставляем его
            no_green_image = sepia_image
        no_green_images.append(no_green_image)

        # Гистограммная эквализация для улучшения контраста
        hist_equalized = exposure.equalize_hist(no_green_image)
        hist_images.append(hist_equalized)

        gauss_image = cv2.GaussianBlur(resized_image, (5, 5), 0)
        gauss_images.append(gauss_image)
        
        # Гауссово размытие для уменьшения шума
        smoothed_image = mh.gaussian_filter(hist_equalized, 3)
        smoothed_images.append(smoothed_image)  # Сохраняем промежуточный результат

        # Убедимся, что смазанное изображение 2D перед передачей в Canny
        if smoothed_image.ndim == 3:
            smoothed_image = smoothed_image[:, :, 0]  # Берем один канал, если изображение 3D (RGB)

        # Обнаруживаем края с помощью Canny перед пороговой фильтрацией
        edges = canny(smoothed_image, sigma=1)  # Убедитесь, что изображение 2D
        
        # Добавляем к списку результатов
        preprocessed_images.append(edges)

        # Пороговая фильтрация с использованием метода Отсу
        threshold = filters.threshold_otsu(smoothed_image)
        binary_image = smoothed_image > threshold
        binary_images.append(binary_image)
        
        # K-Means сегментация (2 кластера)
        img_for_kmeans = smoothed_image  # Можно использовать оригинал или smoothed_image
        pixels = img_for_kmeans.reshape(-1, 1)
        kmeans = KMeans(n_clusters=2, random_state=42).fit(pixels)
        segmented_kmeans = kmeans.labels_.reshape(img_for_kmeans.shape)
        segmented_kmeans_images.append(segmented_kmeans)

        # Выделение объектов по маске
        mask = binary_image.copy()
        mask = morphology.remove_small_objects(mask, min_size=50)
        mask = morphology.remove_small_holes(mask, area_threshold=50)
        labeled = measure.label(mask)
        masked_objects = color.label2rgb(labeled, image=gray_image, bg_label=0)
        masked_objects_images.append(masked_objects)        

    return (np.array(preprocessed_images), np.array(gauss_images), np.array(sepia_images), np.array(normalized_images), 
            np.array(smoothed_images), np.array(binary_images), np.array(segmented_kmeans_images), np.array(masked_objects_images))

## Извлечение признаков

In [25]:
import matplotlib.pyplot as plt
from skimage.feature import hog
from skimage import color, exposure

# Функция для извлечения признаков HOG
def extract_hog_features(processed_images):
    hog_features = []
    hog_images = []  # Для хранения изображений HOG для визуализации

    for image in processed_images:
        # Преобразуем в градации серого (если изображение цветное)
        if len(image.shape) == 3:
            image = color.rgb2gray(image)

        # Вычисляем HOG признаки     
        features, hog_image = hog(image, orientations=9, pixels_per_cell=(16, 16),
                                  cells_per_block=(2, 2), visualize=True, channel_axis=None)

        # Нормализуем изображение HOG для визуализации
        hog_image_rescaled = exposure.rescale_intensity(hog_image, in_range=(0, 10))

        # Добавляем признаки в список
        hog_features.append(features)
        hog_images.append(hog_image_rescaled)
    
    return np.array(hog_features, dtype=object), hog_images  # Возвращаем и признаки, и изображения HOG

# Пример загрузки изображения
from skimage import data

image = data.astronaut()  # Цветное изображение

# Запускаем функцию
features, hog_images = extract_hog_features([image])

In [26]:
from imblearn.over_sampling import SMOTE

# Применяем SMOTE для увеличения меньшего класса
print("Количество классов в y_train:", np.bincount(y_train))

# Использование SMOTE только если есть два класса
if len(np.unique(y_train)) > 1:
    smote = SMOTE(sampling_strategy='auto', random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)
else:
    print("Недостаточно классов в y_train для применения SMOTE")

# Обучаем модель на сбалансированных данных
model.fit(X_train_resampled, y_train_resampled)

# Предсказание
y_pred = model.predict(X_test)

Количество классов в y_train: [30 34 31 33]


## Глубокое обучение на оснве TensorFlow

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
import matplotlib.pyplot as plt
from matplotlib.image import imread
import os
from pathlib import Path
import numpy as np
import pandas as pd
import cv2

Ещё раз получим изображения 

In [28]:
# Пути к директориям
train_dir = "./static/images/X-ray/train"
val_dir = "./static/images/X-ray/test"
PIC_SIZE = 256
BATCH_SIZE = 8

# === Загружаем вручную ===
def load_and_label_images_from_directory(directory, target_size=(256, 256)):
    images = []
    labels = []
    class_names = sorted(os.listdir(directory))

    for class_name in class_names:
        class_path = os.path.join(directory, class_name)
        if not os.path.isdir(class_path):
            continue
        for file in sorted(os.listdir(class_path)):
            if file.lower().endswith((".jpg", ".png", ".jpeg", ".bmp")):
                file_path = os.path.join(class_path, file)
                img = cv2.imread(file_path)
                if img is None:
                    continue
                if img.shape[:2] != target_size:
                    img = cv2.resize(img, target_size)
                images.append(img)
                labels.append(class_name)

    return images, labels

# === Загружаем и предобрабатываем ===
train_images_raw, train_labels_raw = load_and_label_images_from_directory(train_dir, (PIC_SIZE, PIC_SIZE))
val_images_raw, val_labels_raw = load_and_label_images_from_directory(val_dir, (PIC_SIZE, PIC_SIZE))

_, gauss_train, *_= preprocess_images(train_images_raw, target_size=(PIC_SIZE, PIC_SIZE))
_, gauss_val, *_ = preprocess_images(val_images_raw, target_size=(PIC_SIZE, PIC_SIZE))

# Нормализуем изображения
train_images_tensor = gauss_train.astype(np.float32) / 255.0
val_images_tensor = gauss_val.astype(np.float32) / 255.0

# Кодируем метки
label_encoder = LabelEncoder()
train_labels_encoded = to_categorical(label_encoder.fit_transform(train_labels_raw))
val_labels_encoded = to_categorical(label_encoder.transform(val_labels_raw))

# Преобразуем изображения и метки в tf.data.Dataset
def convert_to_grayscale(image, label):
    image = tf.image.rgb_to_grayscale(image)
    return image, label

# Используем from_tensor_slices для передачи признаков и закодированных меток (целевой функции)
train_ds = tf.data.Dataset.from_tensor_slices((train_images_tensor, train_labels_encoded))
val_ds = tf.data.Dataset.from_tensor_slices((val_images_tensor, val_labels_encoded))

# Преобразуем изображения в оттенки серого
train_ds = train_ds.map(convert_to_grayscale)
val_ds = val_ds.map(convert_to_grayscale)

# Перемешиваем и объединяем в батчи
train_ds = train_ds.shuffle(1000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Train dataset and validation dataset successfully prepared.")

print("Train shape:", train_images_tensor.shape)
print("Validation shape:", val_images_tensor.shape)
print("Классы:", label_encoder.classes_)


e:\MII\laboratory\mai\Lib\site-packages\skimage\_shared\utils.py:445: UserWarning: This might be a color image. The histogram will be computed on the flattened image. You can instead apply this function to each color channel, or set channel_axis.
  return func(*args, **kwargs)
e:\MII\laboratory\mai\Lib\site-packages\skimage\_shared\utils.py:445: UserWarning: This might be a color image. The histogram will be computed on the flattened image. You can instead apply this function to each color channel, or set channel_axis.
  return func(*args, **kwargs)
e:\MII\laboratory\mai\Lib\site-packages\skimage\_shared\utils.py:445: UserWarning: This might be a color image. The histogram will be computed on the flattened image. You can instead apply this function to each color channel, or set channel_axis.
  return func(*args, **kwargs)
e:\MII\laboratory\mai\Lib\site-packages\skimage\_shared\utils.py:445: UserWarning: This might be a color image. The histogram will be computed on the flattened image.

Train dataset and validation dataset successfully prepared.
Train shape: (437, 256, 256, 3)
Validation shape: (457, 256, 256, 3)
Классы: ['Covid' 'Normal' 'Viral Pneumonia']


## Настройка гиперпараметров модели

In [29]:
model_simple = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(PIC_SIZE, PIC_SIZE, 1), padding='same', kernel_initializer="he_normal"),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=3, padding='same'),
    
    tf.keras.layers.Conv2D(64, (3,3), activation='relu', padding='same', kernel_initializer="he_normal"),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=3, padding='same'),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu', padding='same', kernel_initializer="he_normal"),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=3, padding='same'),
    
    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu', kernel_initializer="he_normal"),
    tf.keras.layers.Dropout(0.3),
    
    tf.keras.layers.Dense(len(label_encoder.classes_), activation='softmax')
])

e:\MII\laboratory\mai\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


## Компиляция и обучение модели

In [30]:
model_simple.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
#categorical_crossentropy or binary_crossentropy (нужен мультикласс)

# Теперь можно обучать модель
model_simple.fit(train_ds, validation_data=val_ds, epochs=50)

Epoch 1/50
55/55 ━━━━━━━━━━━━━━━━━━━━ 5s 66ms/step - accuracy: 0.5403 - loss: 1.0355 - val_accuracy: 0.6324 - val_loss: 0.9649
Epoch 2/50
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 57ms/step - accuracy: 0.5798 - loss: 0.9304 - val_accuracy: 0.6171 - val_loss: 0.8851
Epoch 3/50
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 57ms/step - accuracy: 0.6708 - loss: 0.7995 - val_accuracy: 0.6018 - val_loss: 0.8143
Epoch 4/50
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.6611 - loss: 0.7690 - val_accuracy: 0.6280 - val_loss: 0.7886
Epoch 5/50
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 58ms/step - accuracy: 0.7139 - loss: 0.6678 - val_accuracy: 0.6477 - val_loss: 0.7175
Epoch 6/50
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.7438 - loss: 0.5599 - val_accuracy: 0.6368 - val_loss: 0.7276
Epoch 7/50
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 56ms/step - accuracy: 0.7938 - loss: 0.5472 - val_accuracy: 0.5996 - val_loss: 0.8042
Epoch 8/50
55/55 ━━━━━━━━━━━━━━━━━━━━ 3s 57ms/step - accuracy: 0.7343 - loss: 0.5392 - val_accuracy: 0.6455 - v

In [31]:
model_simple.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 256, 256, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 86, 86, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 86, 86, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 29, 29, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 29, 29, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 10, 10, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 303,371 (1.16 MB)

 Trainable params: 101,123 (395.01 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 202,248 (790.04 KB)

In [32]:
predict_label = []

# Получаем предсказания для всех изображений в val_ds
y_pred = model_simple.predict(val_ds, batch_size=BATCH_SIZE)

# Берём только первые 40 предсказаний
prediction = y_pred[0:40]

# Итератор по датасету
val_ds_iter = iter(val_ds)

# Собираем 40 изображений
images_list = []
while len(images_list) < 40:
    images, _ = next(val_ds_iter)
    for image in images:
        if len(images_list) >= 40:
            break
        images_list.append(image)


58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step


In [33]:
# Оценка модели на обучающем наборе
train_loss, train_acc = model_simple.evaluate(train_ds)
print(f"Train Loss: {train_loss}, Train Accuracy: {train_acc}")

# Оценка модели на валидационном наборе
val_loss, val_acc = model_simple.evaluate(val_ds)
print(f"Validation Loss: {val_loss}, Validation Accuracy: {val_acc}")


55/55 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9358 - loss: 0.1190
Train Loss: 0.14070139825344086, Train Accuracy: 0.9382150769233704
58/58 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.9236 - loss: 0.4213
Validation Loss: 1.7212371826171875, Validation Accuracy: 0.7111597657203674


In [34]:
deep_model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3,3), activation='relu', input_shape=(PIC_SIZE, PIC_SIZE, 1), kernel_regularizer = tf.keras.regularizers.l2(0.00001), kernel_initializer = "he_normal", strides=1, padding='same'),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=2, padding="same"),
    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Conv2D(64, (3,3), activation='relu', kernel_initializer = "he_normal", padding='same'),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=2, padding='same'),
    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Conv2D(128, (3,3), activation='relu', padding='same', kernel_initializer="he_normal"),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=2, padding='same'),
    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.Conv2D(256, (3,3), activation='relu', padding='same', kernel_initializer="he_normal"),
    tf.keras.layers.MaxPooling2D(pool_size=(2,2), strides=2, padding='same'),
    tf.keras.layers.BatchNormalization(),

    tf.keras.layers.GlobalAveragePooling2D(),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(256, activation='relu'),
    tf.keras.layers.Dropout(0.5),

    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(len(label_encoder.classes_), activation='softmax')
])

In [35]:
deep_model.compile(
    optimizer=tf.keras.optimizers.Nadam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=['acc']
)

history = deep_model.fit(
    train_ds,   
    epochs=20, 
    validation_data=val_ds)

print("Модель натренировалась")

Epoch 1/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 13s 186ms/step - acc: 0.6820 - loss: 0.7737 - val_acc: 0.2035 - val_loss: 2.1362
Epoch 2/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 178ms/step - acc: 0.7596 - loss: 0.5749 - val_acc: 0.1247 - val_loss: 2.0075
Epoch 3/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 174ms/step - acc: 0.8656 - loss: 0.4043 - val_acc: 0.0569 - val_loss: 5.1114
Epoch 4/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 180ms/step - acc: 0.8901 - loss: 0.3628 - val_acc: 0.6346 - val_loss: 1.3418
Epoch 5/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 188ms/step - acc: 0.8435 - loss: 0.4182 - val_acc: 0.5055 - val_loss: 1.6607
Epoch 6/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 177ms/step - acc: 0.9080 - loss: 0.2549 - val_acc: 0.2779 - val_loss: 3.8570
Epoch 7/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 185ms/step - acc: 0.9235 - loss: 0.2444 - val_acc: 0.0678 - val_loss: 3.3456
Epoch 8/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 10s 177ms/step - acc: 0.9075 - loss: 0.2581 - val_acc: 0.6521 - val_loss: 1.9553
Epoch 9/20
55/55 ━━━━━━━━━━━━━━━━━━━━ 11s 191ms/

In [36]:
deep_model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_3 (Conv2D)               │ (None, 256, 256, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 128, 128, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 128, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 128, 128, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 64, 64, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 64, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 64, 64, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 32, 32, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 32, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 32, 32, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 16, 16, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 16, 16, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 256)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 256)            │        65,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 3)              │           387 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,464,588 (5.59 MB)

 Trainable params: 487,875 (1.86 MB)

 Non-trainable params: 960 (3.75 KB)

 Optimizer params: 975,753 (3.72 MB)

In [37]:
# Оценка модели на обучающем наборе
train_loss, train_acc = deep_model.evaluate(train_ds)
print(f"Train Loss: {train_loss}, Train Accuracy: {train_acc}")

# Оценка модели на валидационном наборе
val_loss, val_acc = deep_model.evaluate(val_ds)
print(f"Validation Loss: {val_loss}, Validation Accuracy: {val_acc}")

55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 40ms/step - acc: 0.9468 - loss: 0.0989
Train Loss: 0.09829962998628616, Train Accuracy: 0.9542334079742432
58/58 ━━━━━━━━━━━━━━━━━━━━ 2s 39ms/step - acc: 0.9193 - loss: 0.4602
Validation Loss: 2.019744634628296, Validation Accuracy: 0.7155361175537109


На графике показана история обучения модели: точность (acc, val_acc) и функция потерь (loss, val_loss) по эпохам.  

Точность (Accuracy):  
acc (синяя линия) — точность на обучающем наборе. Почти с самого начала она высокая (~0.95–0.98) и остаётся стабильной.  
val_acc (зелёная линия) — точность на валидации. Также держится стабильно, немного ниже acc, но без серьёзных просадок.  
Вывод: модель хорошо обучается и обобщается на валидации. Отсутствие расхождения между acc и val_acc говорит о том, что переобучение минимально. Обучение проходит успешно. Нет признаков переобучения: val_loss и val_acc не начинают ухудшаться. Можно рассмотреть увеличение числа эпох, если нужно ещё немного улучшить точность, или попробовать регуляризацию / аугментацию, если хочется более надёжной обобщаемости.

In [38]:
test_image_viral = tf.keras.preprocessing.image.load_img("./static/images/X-ray/test/Viral Pneumonia/VIRUS-262135-0001.jpeg", target_size=(256, 256), color_mode='grayscale')

X = tf.keras.preprocessing.image.img_to_array(test_image_viral)
X = np.expand_dims(X, axis = 0)

prediction = np.vstack([X])

result = deep_model.predict(prediction)

print(result)

arg_max_result = np.argmax(result)

if arg_max_result == 0 :
    print("Пациент с ковидлой")
elif arg_max_result == 1 :
    print("Пациент здорового на вид рентгена")
elif arg_max_result == 2 :
    print("Пациент с пневмонией... подхватил всякого разного")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 124ms/step
[[1. 0. 0.]]
Пациент с ковидлой


## Очередная попытка подключить Cuda

In [39]:
!nvidia-smi

!nvcc --version

Sat Apr 19 09:35:06 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 572.61                 Driver Version: 572.61         CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3070 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   50C    P0             26W /   85W |       0MiB /   8192MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Глубокое обучение на оснве PyTorch

In [40]:
import torch 
import torch.nn.functional as F
import IPython.display as ipd
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')

from torch import nn, optim, cuda
from torchvision import transforms, models, datasets
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_curve, auc
from torch.utils.data import DataLoader, TensorDataset

In [41]:
PIC_SIZE = 256

def load_and_label_images_from_directory(directory, target_size=(256, 256)):
    images = []
    labels = []
    class_names = sorted(os.listdir(directory))

    for class_name in class_names:
        class_path = os.path.join(directory, class_name)
        if not os.path.isdir(class_path):
            continue
        for file in sorted(os.listdir(class_path)):
            if file.lower().endswith((".jpg", ".png", ".jpeg", ".bmp")):
                file_path = os.path.join(class_path, file)
                img = cv2.imread(file_path)
                if img is None:
                    continue
                img = cv2.resize(img, target_size)
                images.append(img)
                labels.append(class_name)

    return np.array(images), np.array(labels)

train_dir = "./static/images/X-ray hand/train"
val_dir = "./static/images/X-ray hand/test"

train_images_raw, train_labels_raw = load_and_label_images_from_directory(train_dir)
val_images_raw, val_labels_raw = load_and_label_images_from_directory(val_dir)

_, gauss_train, *_ = preprocess_images(train_images_raw)
_, gauss_val, *_ = preprocess_images(val_images_raw)

# Нормализация и преобразование
train_images_tensor = gauss_train.astype(np.float32) / 255.0
val_images_tensor = gauss_val.astype(np.float32) / 255.0

# Преобразование в grayscale
train_images_tensor = np.mean(train_images_tensor, axis=-1, keepdims=True)
val_images_tensor = np.mean(val_images_tensor, axis=-1, keepdims=True)

# Трансформируем в torch tensors
train_images_tensor = torch.tensor(train_images_tensor).permute(0, 3, 1, 2)
val_images_tensor = torch.tensor(val_images_tensor).permute(0, 3, 1, 2)

# Кодируем метки
label_encoder = LabelEncoder()
train_labels_encoded = torch.tensor(label_encoder.fit_transform(train_labels_raw), dtype=torch.long)
val_labels_encoded = torch.tensor(label_encoder.transform(val_labels_raw), dtype=torch.long)

# Создаем датасеты
train_dataset = TensorDataset(train_images_tensor, train_labels_encoded)
val_dataset = TensorDataset(val_images_tensor, val_labels_encoded)

BATCH_SIZE = 8
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print(label_encoder.classes_)

['damage' 'no damage']


In [42]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv_layers = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.fc_layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear((PIC_SIZE // 4) * (PIC_SIZE // 4) * 64, 128),
            nn.ReLU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        return self.fc_layers(x)


In [43]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_classes = len(label_encoder.classes_)
model = SimpleCNN(num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

train_loss_history = []
train_acc_history = []

EPOCHS = 30
for epoch in range(EPOCHS):
    model.train()
    running_loss, correct = 0.0, 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct / len(train_dataset)

    train_loss_history.append(epoch_loss)
    train_acc_history.append(epoch_acc)

    acc = correct / len(train_dataset)
    print(f"[{epoch+1}/{EPOCHS}] Loss: {running_loss:.4f} | Accuracy: {acc:.4f}")



[1/30] Loss: 55.7976 | Accuracy: 0.6667
[2/30] Loss: 30.2475 | Accuracy: 0.6111
[3/30] Loss: 16.4770 | Accuracy: 0.7778
[4/30] Loss: 5.8538 | Accuracy: 0.9444
[5/30] Loss: 2.1740 | Accuracy: 1.0000
[6/30] Loss: 0.2372 | Accuracy: 1.0000
[7/30] Loss: 0.1248 | Accuracy: 1.0000
[8/30] Loss: 0.0199 | Accuracy: 1.0000
[9/30] Loss: 0.0042 | Accuracy: 1.0000
[10/30] Loss: 0.0009 | Accuracy: 1.0000
[11/30] Loss: 0.0006 | Accuracy: 1.0000
[12/30] Loss: 0.0005 | Accuracy: 1.0000
[13/30] Loss: 0.0004 | Accuracy: 1.0000
[14/30] Loss: 0.0004 | Accuracy: 1.0000
[15/30] Loss: 0.0004 | Accuracy: 1.0000
[16/30] Loss: 0.0004 | Accuracy: 1.0000
[17/30] Loss: 0.0003 | Accuracy: 1.0000
[18/30] Loss: 0.0003 | Accuracy: 1.0000
[19/30] Loss: 0.0003 | Accuracy: 1.0000
[20/30] Loss: 0.0003 | Accuracy: 1.0000
[21/30] Loss: 0.0003 | Accuracy: 1.0000
[22/30] Loss: 0.0003 | Accuracy: 1.0000
[23/30] Loss: 0.0003 | Accuracy: 1.0000
[24/30] Loss: 0.0003 | Accuracy: 1.0000
[25/30] Loss: 0.0003 | Accuracy: 1.0000
[26/30

In [44]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        predicted = outputs.argmax(1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

print(f"Validation Accuracy: {correct / total:.4f}")


Validation Accuracy: 1.0000


In [46]:
torch.cuda.is_available()

False